# 气候 Claim 分类 — 正式端到端管线

**流程**：数据 → **BM25 top-50** ∪ **embedding top-50 中不在 BM25 的前 20 条** → Cross-encoder 微调 → 加载 CE、`retrieve_reranked` → **结构化** `build_dataset` → **DistilBERT + LoRA**（`lr3e-05_ep3_ml512`）→ dev/test 预测 → dev 评测。

自上而下顺序运行；首次运行需联网下载句向量、CE 与 **peft**。


In [1]:
# 安装到当前 notebook 内核
%pip install -q "peft>=0.11.0" rank_bm25 transformers datasets scikit-learn tqdm torch numpy pandas scipy accelerate sentence-transformers



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json

with open("data/train-claims.json") as f:
    train_data = json.load(f)

with open("data/dev-claims.json") as f:
    dev_data = json.load(f)

with open("data/evidence.json") as f:
    evidence_data = json.load(f)

print("train claims:", len(train_data), "| dev:", len(dev_data), "| evidence:", len(evidence_data))


train claims: 1228 | dev: 154 | evidence: 1208827


## 1. 检索索引（BM25 + 句向量 embedding；CE 候选 = **BM25@50 ∪ (embedding@50 中不在 BM25 的前 20)**）


In [5]:
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

eid_list = list(evidence_data.keys())
corpus = list(evidence_data.values())

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMB_ENCODE_BATCH = 256
_emb_cache_dir = Path("data/embedding_cache")
_emb_cache_dir.mkdir(parents=True, exist_ok=True)
_corpus_emb_path = _emb_cache_dir / "corpus_embeddings.npy"

_emb_device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=_emb_device)
print("SentenceTransformer:", EMBEDDING_MODEL_NAME, "| device:", _emb_device)

if _corpus_emb_path.is_file():
    corpus_embeddings = np.load(_corpus_emb_path)
    if corpus_embeddings.shape[0] != len(eid_list):
        raise RuntimeError(
            f"缓存行数 {corpus_embeddings.shape[0]} 与 evidence 条数 {len(eid_list)} 不一致，请删 {_corpus_emb_path} 后重跑本单元。"
        )
    print("Loaded corpus embeddings:", _corpus_emb_path, corpus_embeddings.shape)
else:
    corpus_embeddings = embed_model.encode(
        corpus,
        batch_size=EMB_ENCODE_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(_corpus_emb_path, corpus_embeddings)
    print("Saved corpus embeddings:", _corpus_emb_path, corpus_embeddings.shape)


def _retrieve_embedding_topk(claim, k=50):
    # L2-normalized vectors: dot product = cosine similarity; return top-k eids high-to-low.
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    q = embed_model.encode(
        [claim],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].astype(np.float32)
    scores = corpus_embeddings @ q
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10032.12it/s]


SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2 | device: mps


Batches:   2%|▏         | 87/4722 [00:51<45:52,  1.68it/s]  


KeyboardInterrupt: 

In [ ]:
import re
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def _tokenize_bm25(text):
    text = (text or "").lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    stop = ENGLISH_STOP_WORDS
    return [t for t in tokens if t not in stop]


tokenized_corpus = [_tokenize_bm25(doc) for doc in tqdm(corpus, desc="Tokenize for BM25")]
tokenized_corpus = [t if t else ["_"] for t in tokenized_corpus]
bm25 = BM25Okapi(tokenized_corpus)


def _retrieve_bm25_topk(claim, k=50):
    n = len(eid_list)
    k = min(k, n)
    if k == 0:
        return []
    q_tokens = _tokenize_bm25(claim)
    if not q_tokens:
        q_tokens = ["_"]
    scores = bm25.get_scores(q_tokens)
    top_k_idx = np.argpartition(scores, -k)[-k:]
    top_k_idx = top_k_idx[np.argsort(scores[top_k_idx])[::-1]]
    return [eid_list[i] for i in top_k_idx]


retrieve_top_k_fast = _retrieve_bm25_topk


## 2. Cross-encoder 微调（train：**BM25@50 ∪ emb 独有前 20** 并集内 gold / 难负+随机负）

微调单元在 `old_fit` **前后**各在 **dev** 上算一次 **mean gold rank in U**（仅统计落在候选 U 内的 gold；CE 分数越高 rank 越靠前为 1）。


In [ ]:
from tqdm import tqdm

BM25_TOP_K = 50
EMBEDDING_TOP_K = 50
EMB_EXTRA_NOT_IN_BM25 = 20


def union_bm25_embedding_eids(bm_eids, emb_ordered_eids, n_extra=EMB_EXTRA_NOT_IN_BM25):
    bm_set = set(bm_eids)
    tail = [e for e in emb_ordered_eids if e not in bm_set][:n_extra]
    return list(dict.fromkeys(list(bm_eids) + tail))


if "_retrieve_embedding_topk" not in globals():
    raise RuntimeError("请先运行上方句向量 embedding 索引单元。")
if "_retrieve_bm25_topk" not in globals():
    raise RuntimeError("请先运行上方 BM25 索引单元。")

train_union_cache = {}
n_claims = n_gold_links = n_gold_in_u = n_non_gold = 0

for cid, item in tqdm(train_data.items(), desc="train: U, gold, U\gold"):
    gold = set(item.get("evidences") or [])
    if not gold:
        continue
    claim = item["claim_text"]
    bm = _retrieve_bm25_topk(claim, k=BM25_TOP_K)
    emb = _retrieve_embedding_topk(claim, k=EMBEDDING_TOP_K)
    U = union_bm25_embedding_eids(bm, emb, n_extra=EMB_EXTRA_NOT_IN_BM25)
    gold_in_U = [e for e in U if e in gold]
    non_gold = [e for e in U if e not in gold]
    train_union_cache[cid] = {
        "claim": claim,
        "gold": gold,
        "U": U,
        "gold_in_U": gold_in_U,
        "non_gold": non_gold,
    }
    n_claims += 1
    n_gold_links += len(gold)
    n_gold_in_u += len(gold_in_U)
    n_non_gold += len(non_gold)

print(f"Train claims with gold: {n_claims}")
print(f"Total gold evidence links: {n_gold_links}")
print(f"Gold in union |G∩U|: {n_gold_in_u}  (recall vs all gold links: {n_gold_in_u / n_gold_links:.4f})")
print(f"Non-gold slots in U (sum over claims): {n_non_gold}")


In [ ]:
import random

import numpy as np
import torch
from sentence_transformers import CrossEncoder, InputExample
from tqdm import tqdm

NUM_HARD_NEG = 8
NUM_RANDOM_NEG = 2
CE_MINE_BATCH = 32
CROSS_ENCODER_BASE = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if "train_union_cache" not in globals():
    raise RuntimeError("请先运行上一单元生成 train_union_cache。")

_ce_mine_kw = {}
if torch.cuda.is_available():
    _ce_mine_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_mine_kw["device"] = "mps"
ce_miner = CrossEncoder(CROSS_ENCODER_BASE, **_ce_mine_kw)
print("Hard-negative miner device:", _ce_mine_kw.get("device", "cpu (default)"))

rng = random.Random(42)


def pick_hard_negatives(claim, non_gold_eids, k):
    if not non_gold_eids or k <= 0:
        return []
    k = min(k, len(non_gold_eids))
    pairs = [(claim, evidence_data[eid]) for eid in non_gold_eids]
    scores = ce_miner.predict(pairs, show_progress_bar=False, batch_size=CE_MINE_BATCH)
    order = np.argsort(scores)[::-1][:k]
    return [non_gold_eids[i] for i in order]


ce_train_examples = []
n_pos = n_neg = 0

for cid, rec in tqdm(train_union_cache.items(), desc="build CE train pairs"):
    claim = rec["claim"]
    for eid in rec["gold_in_U"]:
        ce_train_examples.append(
            InputExample(texts=[claim, evidence_data[eid]], label=1.0)
        )
        n_pos += 1
    non_gold = rec["non_gold"]
    if not non_gold:
        continue
    hard = pick_hard_negatives(claim, non_gold, NUM_HARD_NEG)
    hard_set = set(hard)
    pool = [e for e in non_gold if e not in hard_set]
    rand_k = min(NUM_RANDOM_NEG, len(pool))
    easy = rng.sample(pool, rand_k) if rand_k else []
    for eid in hard + easy:
        ce_train_examples.append(
            InputExample(texts=[claim, evidence_data[eid]], label=0.0)
        )
        n_neg += 1

print(f"Examples: {len(ce_train_examples)}  (positive={n_pos}, negative={n_neg})")


In [ ]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder
from tqdm import tqdm

if "ce_train_examples" not in globals() or not ce_train_examples:
    raise RuntimeError("请先运行构造 ce_train_examples 的单元。")
if "dev_data" not in globals():
    raise RuntimeError("请先运行加载数据的单元（dev_data）。")
if "_retrieve_bm25_topk" not in globals() or "_retrieve_embedding_topk" not in globals():
    raise RuntimeError("请先运行 BM25 与 embedding 索引单元。")

CROSS_ENCODER_BASE = globals().get(
    "CROSS_ENCODER_BASE", "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()
CE_EPOCHS = 1
CE_BATCH_SIZE = 16
CE_LR = 2e-5
CE_RANK_BATCH = 32

BM25_K = int(globals().get("BM25_TOP_K", 50))
EMB_K = int(globals().get("EMBEDDING_TOP_K", 50))
EMB_EXTRA = int(globals().get("EMB_EXTRA_NOT_IN_BM25", 20))


def _candidate_u_for_claim(claim):
    bm = _retrieve_bm25_topk(claim, k=BM25_K)
    emb = _retrieve_embedding_topk(claim, k=EMB_K)
    uni = globals().get("union_bm25_embedding_eids")
    if callable(uni):
        return uni(bm, emb, n_extra=EMB_EXTRA)
    bm_set = set(bm)
    tail = [e for e in emb if e not in bm_set][:EMB_EXTRA]
    return list(dict.fromkeys(list(bm) + tail))


# Gold in U only: 1-based rank by CE score within U (1 = best).
def mean_gold_rank_in_u(ce_model, claims_data, batch_size=CE_RANK_BATCH):
    ranks = []
    for cid, item in tqdm(claims_data.items(), desc="mean gold rank in U (dev)"):
        gold = set(item.get("evidences") or [])
        if not gold:
            continue
        claim = item["claim_text"]
        U = _candidate_u_for_claim(claim)
        if not U:
            continue
        pairs = [(claim, evidence_data[eid]) for eid in U]
        scores = ce_model.predict(pairs, show_progress_bar=False, batch_size=batch_size)
        order = np.argsort(scores)[::-1]
        ranked = [U[i] for i in order]
        pos = {eid: r + 1 for r, eid in enumerate(ranked)}
        for eid in gold:
            if eid in pos:
                ranks.append(pos[eid])
    if not ranks:
        return float("nan"), 0
    return float(np.mean(ranks)), len(ranks)


ce_train_dataloader = DataLoader(
    ce_train_examples, shuffle=True, batch_size=CE_BATCH_SIZE
)

cross_encoder = CrossEncoder(
    CROSS_ENCODER_BASE,
    num_labels=1,
    **_ce_mine_kw,
)

m_before, n_before = mean_gold_rank_in_u(cross_encoder, dev_data)
print(
    "[CE before fine-tune] mean gold rank in U (dev, ranks for gold in U only): "
    f"{m_before:.4f}  (n={n_before})"
)

_param_snap = {
    n: p.detach().cpu().clone()
    for n, p in cross_encoder.model.named_parameters()
}

warmup = min(100, max(1, len(ce_train_examples) // CE_BATCH_SIZE))
cross_encoder.old_fit(
    train_dataloader=ce_train_dataloader,
    epochs=CE_EPOCHS,
    warmup_steps=warmup,
    optimizer_params={"lr": CE_LR},
    show_progress_bar=True,
)

m_after, n_after = mean_gold_rank_in_u(cross_encoder, dev_data)
print(
    "[CE after fine-tune]  mean gold rank in U (dev, ranks for gold in U only): "
    f"{m_after:.4f}  (n={n_after})"
)
if n_before == n_after and n_before > 0:
    print(f"  delta (after - before): {m_after - m_before:+.4f}  (lower is better)")

_max_delta = max(
    (p.detach().cpu() - _param_snap[n]).abs().max().item()
    for n, p in cross_encoder.model.named_parameters()
)
print(f"Max |param_after - param_before|: {_max_delta:.6e}")
if _max_delta < 1e-7:
    raise RuntimeError("训练后参数几乎未变，请勿 save；检查 device / loss / dataloader。")

CE_FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
cross_encoder.save(str(CE_FINETUNE_DIR))
print("Fine-tuned CrossEncoder saved to:", CE_FINETUNE_DIR)


## 3. 正式管线：加载 CE + `retrieve_reranked` + `build_dataset`


In [ ]:
import numpy as np
import torch
from sentence_transformers import CrossEncoder
from pathlib import Path

BM25_TOP_K = 50
EMBEDDING_TOP_K = 50
EMB_EXTRA_NOT_IN_BM25 = 20

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_FINETUNE_DIR = Path("ce_finetuned").resolve()

_ce_kw = {}
if torch.cuda.is_available():
    _ce_kw["device"] = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    _ce_kw["device"] = "mps"

_ce_has_finetuned = CE_FINETUNE_DIR.is_dir() and (CE_FINETUNE_DIR / "config.json").is_file()
_ce_load_path = str(CE_FINETUNE_DIR) if _ce_has_finetuned else CROSS_ENCODER_MODEL
print("CrossEncoder load:", _ce_load_path, "(finetuned)" if _ce_has_finetuned else "(base)")
cross_encoder = CrossEncoder(_ce_load_path, **_ce_kw)


def union_bm25_embedding_eids(bm_eids, emb_ordered_eids, n_extra=EMB_EXTRA_NOT_IN_BM25):
    bm_set = set(bm_eids)
    tail = [e for e in emb_ordered_eids if e not in bm_set][:n_extra]
    return list(dict.fromkeys(list(bm_eids) + tail))


def retrieve_reranked(claim, retrieve_k=50, final_k=5):
    # Pool: BM25@retrieve_k then up to EMB_EXTRA_NOT_IN_BM25 from embedding@retrieve_k not in BM25 set.
    if "_retrieve_embedding_topk" not in globals():
        raise RuntimeError("缺少 _retrieve_embedding_topk：请先运行 embedding 索引单元。")
    bm_eids = _retrieve_bm25_topk(claim, k=retrieve_k)
    emb_eids = _retrieve_embedding_topk(claim, k=retrieve_k)
    cand_eids = union_bm25_embedding_eids(bm_eids, emb_eids, n_extra=EMB_EXTRA_NOT_IN_BM25)
    if not cand_eids:
        return []
    pairs = [(claim, evidence_data[eid]) for eid in cand_eids]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(scores)[::-1][:final_k]
    return [cand_eids[i] for i in order]


import inspect

print("[OK] BM25+embedding rerank:", inspect.signature(retrieve_reranked))


In [ ]:
from tqdm import tqdm


def format_distilbert_structured_input(claim, ranked_eids, evidence_dict):
    # Single string: claim block + per-evidence blocks (CE order, 1-based rank).
    parts = [f"[CLAIM]{claim}[/CLAIM]"]
    for rank, eid in enumerate(ranked_eids, start=1):
        body = evidence_dict.get(eid, "")
        parts.append(f"[EVIDENCE rank={rank} id={eid}]{body}[/EVIDENCE]")
    return " ".join(parts)


def build_dataset(data, retrieve_k=50, final_k=5):
    texts, labels = [], []
    label_map = {
        "SUPPORTS": 0,
        "REFUTES": 1,
        "NOT_ENOUGH_INFO": 2,
        "DISPUTED": 3,
    }
    for cid, item in tqdm(data.items(), desc="build_dataset"):
        claim = item["claim_text"]
        label = item["claim_label"]
        eids = retrieve_reranked(claim, retrieve_k=retrieve_k, final_k=final_k)
        texts.append(format_distilbert_structured_input(claim, eids, evidence_data))
        labels.append(label_map[label])
    return texts, labels


# Example (fictional ids; same template as real data)
_demo_claim = "Global mean sea level has been rising over recent decades."
_demo_eids = ["demo_ev_1", "demo_ev_2"]
_demo_ev = {
    "demo_ev_1": "Satellite altimetry indicates about 3.4 mm/yr rise since 1993.",
    "demo_ev_2": "Tide gauges and ocean heat content both align with increased water volume.",
}
print("--- Structured DistilBERT input (example) ---")
print(format_distilbert_structured_input(_demo_claim, _demo_eids, _demo_ev))
print("--- end example ---")


In [ ]:
train_texts, train_labels = build_dataset(train_data)
dev_texts, dev_labels = build_dataset(dev_data)


## 4. DistilBERT + **LoRA**（`lr3e-05_ep3_ml512`）

- **输入**：`[CLAIM]…[/CLAIM] [EVIDENCE rank=r id=eid]…[/EVIDENCE]`（与 `build_dataset` / `predict` 一致）。
- **LoRA**：`q_lin` / `k_lin` / `v_lin`；并 `modules_to_save` **分类头** `pre_classifier`、`classifier`（避免只训 adapter 时头被冻死）。


In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import f1_score
from torch.utils.data import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)

if "train_texts" not in globals():
    raise RuntimeError("请先运行 build_dataset 单元。")
if "format_distilbert_structured_input" not in globals():
    raise RuntimeError("请先运行定义 format_distilbert_structured_input 的单元。")

BACKBONE = "distilbert-base-uncased"
LEARNING_RATE = 3e-5
NUM_TRAIN_EPOCHS = 3
MAX_LENGTH = 512
RUN_DIR = Path("distilbert_hyperparam_sweep/lr3e-05_ep3_ml512")
SAVE_DIR = RUN_DIR / "saved_model"
RUN_DIR.mkdir(parents=True, exist_ok=True)

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05


class ClaimDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx]),
        }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float((preds == labels).mean()),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
    }


tokenizer = DistilBertTokenizer.from_pretrained(BACKBONE)
base_model = DistilBertForSequenceClassification.from_pretrained(BACKBONE, num_labels=4)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_lin", "k_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

train_ds = ClaimDataset(train_texts, train_labels, tokenizer, max_len=MAX_LENGTH)
dev_ds = ClaimDataset(dev_texts, dev_labels, tokenizer, max_len=MAX_LENGTH)

use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
print("Trainer device:", "CUDA" if use_cuda else ("MPS" if use_mps else "CPU"))

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / "trainer_output"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    dataloader_pin_memory=use_cuda,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate()
print("Dev metrics:", metrics)

SAVE_DIR.mkdir(parents=True, exist_ok=True)
# PEFT：显式 save_pretrained，避免部分版本 trainer.save_model 未写好 adapter
model.save_pretrained(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))

row = {
    "run_name": "lr3e-05_ep3_ml512_lora",
    "save_dir": str(SAVE_DIR.resolve()),
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_length": MAX_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
}
row.update({k: float(v) for k, v in metrics.items()})
with open(RUN_DIR / "metrics.json", "w") as f:
    json.dump(row, f, indent=2)

DISTILBERT_MAX_LEN_FOR_PREDICT = MAX_LENGTH
print("Saved LoRA adapter + tokenizer ->", SAVE_DIR)

del trainer, train_ds, dev_ds
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()
elif use_mps:
    torch.mps.empty_cache()


## 5. 预测（dev / test）

默认沿用上一节内存里的 `model` / `tokenizer`。若 **重启 kernel** 后只推理：把下一格里的 `RELOAD_FROM_SAVE = True`，会从 `saved_model` 加载 **基座 + LoRA**。


In [ ]:
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from peft import PeftModel

# 重启 kernel 后只跑推理时改为 True
RELOAD_FROM_SAVE = False
SAVE_DIR = Path("distilbert_hyperparam_sweep/lr3e-05_ep3_ml512/saved_model")
BACKBONE = "distilbert-base-uncased"

if RELOAD_FROM_SAVE:
    tokenizer = DistilBertTokenizer.from_pretrained(str(SAVE_DIR))
    _base = DistilBertForSequenceClassification.from_pretrained(BACKBONE, num_labels=4)
    model = PeftModel.from_pretrained(_base, str(SAVE_DIR))
else:
    if "model" not in globals() or "tokenizer" not in globals():
        raise RuntimeError("请先训练上一节，或设置 RELOAD_FROM_SAVE=True")

use_cuda = torch.cuda.is_available()
use_mps = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
device = torch.device("cuda" if use_cuda else ("mps" if use_mps else "cpu"))
print("Predict device:", device)

model.to(device)
model.eval()

label_map_rev = {
    0: "SUPPORTS",
    1: "REFUTES",
    2: "NOT_ENOUGH_INFO",
    3: "DISPUTED",
}


def predict(data, retrieve_k=50, final_k=5):
    results = {}
    for cid, item in tqdm(data.items(), desc="predict"):
        claim = item["claim_text"]
        eids = retrieve_reranked(claim, retrieve_k=retrieve_k, final_k=final_k)
        input_text = format_distilbert_structured_input(claim, eids, evidence_data)
        max_len = int(globals().get("DISTILBERT_MAX_LEN_FOR_PREDICT", 512))
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_len,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        logits = outputs.logits.detach().float().cpu().numpy()
        pred = int(np.argmax(logits))
        results[cid] = {"claim_label": label_map_rev[pred], "evidences": eids}
    return results


In [ ]:
import json

dev_predictions = predict(dev_data)
with open("dev_predictions.json", "w") as f:
    json.dump(dev_predictions, f, indent=2)
print("Wrote dev_predictions.json")


In [ ]:
import json

with open("data/test-claims-unlabelled.json") as f:
    test_data = json.load(f)
test_predictions = predict(test_data)
with open("test_predictions.json", "w") as f:
    json.dump(test_predictions, f, indent=2)
print("Wrote test_predictions.json")


## 6. Evaluation（dev）

对 **`dev_predictions.json`** 与 **`data/dev-claims.json`** 计算：证据检索 F-score（F）、claim 分类准确率（A）、F 与 A 的调和平均（与课程评测脚本一致）。先显式读入 JSON，再构造 `EvalArgs`。

In [ ]:
class Args:

  def __init__(self):

    self.predictions = "dev_predictions.json"
    # self.predictions = "test_predictions.json"

    # self.predictions = "data/dev-claims-baseline.json"

    self.groundtruth = "data/dev-claims.json"

    self.verbose = True

In [ ]:
import argparse
import sys
import json
import numpy as np

######
#main#
######

def main(args):

    try:
        predictions = json.load(open(args.predictions))
    except:
        print("Error loading predictions json file:", args.predictions)
        raise SystemExit

    try:
        groundtruth = json.load(open(args.groundtruth))
    except:
        print("Error loading groundtruth json file:", args.groundtruth)
        raise SystemExit

    try:
        f, acc = [], []

        #iterate through the groundtruth instances
        for claim_id, claim in sorted(groundtruth.items()):
            if claim_id in predictions and \
                "claim_label" in predictions[claim_id] and \
                "evidences" in predictions[claim_id]:

                #check claim level label
                instance_correct = 0.0
                if predictions[claim_id]["claim_label"] == claim["claim_label"]:
                    instance_correct = 1.0

                #check retrieved evidences
                evidence_correct = 0
                evidence_recall = 0.0
                evidence_precision = 0.0
                evidence_fscore = 0.0
                if type(predictions[claim_id]["evidences"]) == list and (len(predictions[claim_id]["evidences"]) > 0):
                    top_six_ev = set(predictions[claim_id]["evidences"])
                    for gr_ev in claim["evidences"]:
                        if gr_ev in top_six_ev:
                            evidence_correct += 1
                    if evidence_correct > 0:
                        evidence_recall = float(evidence_correct) / len(claim["evidences"])
                        evidence_precision = \
                            float(evidence_correct) / len(predictions[claim_id]["evidences"])
                        evidence_fscore = (2*evidence_precision*evidence_recall)/(evidence_precision+evidence_recall)

                if args.verbose:
                    print("groundtruth =", claim)
                    print("predictions =", predictions[claim_id])
                    print("instance accuracy =", instance_correct)
                    print("evidence recall =", evidence_recall)
                    print("evidence precision =", evidence_precision)
                    print("evidence fscore =", evidence_fscore, "\n\n")

                #add the metric results
                acc.append(instance_correct)
                f.append(evidence_fscore)

        #compute aggregate performance
        mean_f = np.mean(f if len(f) > 0 else [0.0])
        mean_acc = np.mean(acc if len(acc) > 0 else [0.0])
        if mean_f == 0.0 and mean_acc == 0.0:
            hmean = 0.0
        else:
            hmean = (2*mean_f*mean_acc)/(mean_f+mean_acc)

        print("Evidence Retrieval F-score (F)    =", mean_f)
        print("Claim Classification Accuracy (A) =", mean_acc)
        print("Harmonic Mean of F and A          =", hmean)

    except Exception as error:
        print("Error:", error)
        raise SystemExit

if __name__ == "__main__":

    #parser arguments
    desc = "Evaluation script that computes evidence retrieval f-score, claim classification accuracy, and aggregate performance."
    parser = argparse.ArgumentParser(description=desc)

    #arguments
    # parser.add_argument("--predictions", required=True, help="json file containing the claim label predictions and retrieved evidences produced by a system")
    # parser.add_argument("--groundtruth", required=True, help="json file containing the ground truth claim labels and evidences")
    # parser.add_argument("--verbose", action="store_true", help="turn on debug prints")
    # args = parser.parse_args()

    args = Args()

    main(args)